# CopyLane sLLM(재작성 생성) 파인튜닝 — 로컬 2차 (Qwen2.5-1.5B-Instruct + LoRA)

**이어받는 문서**: `docs/lse/sLLM_파인튜닝_기획서_2026-09-17.md`

- §7의 **Colab 1차 프로토타입**(35쌍 직접 작성 데이터, loss 1.44→0.87, held-out 5/5)을
  이어받는다. 1차는 §3(`transform_pair`/의결서 시정문)이 막혀 교정 문구를 **직접 작성**한
  절충안이었다 — 그 한계가 이번에 풀린다.
- **2차 데이터 구성 — 두 출처를 섞었다** (`docs/lse/train_pairs.jsonl`, 806쌍: train 686 ·
  held_out 120):
  1. **`injected_golden.jsonl`(784행)** — `TRANSFORM_RULES` T1/T2/T4/T5/T6/T7a/T7b로 실제
     승인 문구(118건 중 112건, 나머지 60건은 평가용으로 봉인돼 있어 못 씀)에서 파생시킨
     위반 문구. ⚠️ **틀 다양성이 낮다** — T2(44개 고유 틀) 빼고는 규칙마다 문장 틀이
     3~5종류뿐이라, 모델이 위반을 알아보는 게 아니라 특정 말버릇만 외울 위험이 있다
     (`preprocess/inject.py` 원작성자가 코드 주석에 이미 이 한계를 적어뒀다).
  2. **`ftc_stage.jsonl`의 실제 FTC 의결서 22건(추가)** — 진짜 사건의 인용 위반 문구 +
     시정·금지 명령문에서 뽑았다. 명령문(`주문_마스킹`)은 "~다시 하여서는 아니 된다"류
     금지문이지 재작성된 카피가 아니라서, `RewriteSet.body`는 각 사건 명령문을 **Claude가
     직접 읽고 작성**했다 — 1차 프로토타입의 "교정 문구만 직접 작성"과 같은 성격의 합성이고
     법률 검토를 받지 않았다(`correction_confidence: heuristic_unverified`로 표시). 전체
     후보 285건 중 도메인(화장품·건강기능식품 등)+명확한 반박 근거(퍼센트·최상급 주장)가
     있는 것만 골랐고, 나머지는 상호명뿐이거나 도메인이 달라 제외했다.
- **스키마 갭**: 두 출처 모두 교정문이 단일 텍스트라 `RewriteSet`(body/mandatory_note/
  placement) 중 `body`만 채웠다 — `mandatory_note`/`placement`는 `null`로 두고 모델이
  그렇게 생성하도록 학습한다. 3필드 동시 생성은 다음 데이터 확보 후 과제로 남긴다.
- **로컬 실행**: 1차는 로컬(8GB RAM·GPU 없음)이 부적합해 Colab T4를 썼다 — 이번엔 로컬에
  RTX 3080(10GB VRAM, Ampere)이 확인돼 로컬로 옮긴다. Colab 전용 `google.colab.files.upload()`
  셀은 없다 — `train_pairs.jsonl`을 이 노트북과 같은 폴더에 두면 바로 읽는다.
- bf16 지원 여부는 하드코딩하지 않고 셀에서 `torch.cuda.is_bf16_supported()`로 자동
  판별해 미지원이면 fp16으로 낮춘다 (RTX 3080은 Ampere라 bf16 지원 — 자동판별이라 다른
  GPU로 옮겨도 그대로 동작).


## 0. 환경 설정 (로컬 — 전용 venv)

**이 노트북은 프로젝트 `.venv`가 아니라 `.venv-sllm` 커널로 실행한다.** 프로젝트 `.venv`의
torch는 CPU 빌드(`app/encoder.py`가 쓰는 KC-BERT는 CPU로 충분)라 GPU를 전혀 못 쓴다. CUDA
torch(~2-3GB)를 그 공용 `.venv`에 얹으면 `uv sync` 때 되돌아가고 팀 lock(D-51)과도 충돌하므로,
학습 전용 `.venv-sllm`을 따로 만들어 거기에만 설치했다 — `pyproject.toml`/`uv.lock`은 건드리지
않는다.

Jupyter에서 커널을 `.venv-sllm\Scripts\python.exe`로 선택하고 실행할 것. (아직 `.venv-sllm`이
없다면: `py -3.11 -m venv .venv-sllm` 후 아래 설치 커맨드를 그 venv의 pip로 실행)

In [ ]:
# .venv-sllm 커널에서 실행 중인지 확인 — 프로젝트 공용 .venv면 여기서 멈춘다.
import sys
assert "venv-sllm" in sys.prefix, (
    f"공용 .venv가 아니라 .venv-sllm 커널을 선택하세요 (현재: {sys.prefix})"
)

# torch는 CUDA 빌드(cu128)로 이 venv에 이미 설치돼 있다는 전제 — 없으면 먼저
# python -m pip install torch --index-url https://download.pytorch.org/whl/cu128
%pip install -q transformers accelerate peft


In [ ]:
import json
import random
from pathlib import Path

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

# 🚨 bf16 하드코딩 금지 — GPU가 미지원이면 여기서 자동으로 fp16으로 내려간다.
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("bf16 supported:", BF16_OK, "-> DTYPE:", DTYPE)


## 1. 데이터 로드 — `train_pairs.jsonl`

같은 폴더의 `train_pairs.jsonl`(821쌍: train 700 · held_out 121)을 읽는다. 필드:

- `input`: 위반 문구(모델 입력)
- `violation_types` / `rule_id` / `legal_basis`: 위반 유형·근거 (프롬프트 컨텍스트로 사용)
- `output`: `{"body", "mandatory_note", "placement"}` — `RewriteSet` 스키마, 지금은 `body`만 실값
- `provenance`: `mfds_hf_ingredient_board`(규칙 기반 주입, 784건, 틀 다양성 낮음) 또는
  `ftc_stage`(실제 FTC 의결서, 37건 — `correction_confidence: heuristic_unverified`).
  `ftc_stage` 37건은 두 라운드로 모았다: 1차 22건은 "OOO% 허위" 패턴 위주(교정 전략이
  획일적), 2차 15건은 §7에서 발견한 과적합("OOO에 도움을 줄 수 있음" 고정 꼬리표
  학습)을 깨려고 **의도적으로 다른 교정 전략**(최상급 삭제·절대부정 완화·무첨가
  표시 정정·부당비교 제거 등)이 필요한 사례로 골랐다.
- `split`: `train` / `held_out` (src 단위로 분리 — 데이터 누수 없음)


In [ ]:
DATA_PATH = Path("train_pairs.jsonl")  # 노트북과 같은 폴더
assert DATA_PATH.exists(), f"{DATA_PATH} 가 없다 — 노트북과 같은 폴더에 두었는지 확인"

rows = [json.loads(line) for line in DATA_PATH.open(encoding="utf-8") if line.strip()]

# 🔒 2026-10-01 — 공정위 의결서 원천(ftc_stage) 행은 광고주 문구를 인용해 **재배포 불가**다.
#    공개 저장소에 두지 않고 git 이 무시하는 `_private/ftc_pairs.jsonl` 에 둔다(핵심주장 포함).
#    팀 비공개 데이터 저장소에서 받아 같은 자리에 둔다. 없으면 ftc_stage 없이 학습한다.
PRIVATE_PATH = Path("_private/ftc_pairs.jsonl")
if PRIVATE_PATH.exists():
    rows += [json.loads(line) for line in PRIVATE_PATH.open(encoding="utf-8") if line.strip()]
else:
    print(f"⚠️ {PRIVATE_PATH} 가 없다 — ftc_stage 행 없이 진행한다")
train_rows = [r for r in rows if r["split"] == "train"]
held_out_rows = [r for r in rows if r["split"] == "held_out"]
print(f"train: {len(train_rows)}  held_out: {len(held_out_rows)}  total: {len(rows)}")

# 🔴 §7(1차 실행)에서 과적합을 직접 확인했다 — injected_golden(틀고정, T1 등 3~5틀)이
#    train의 95% 이상을 차지해, ftc_stage 실사례(서로 다른 교정 전략)가 묻혔다.
#    학습 시에만 ftc_stage 행을 OVERSAMPLE 한다 — held_out은 손대지 않는다(평가 왜곡 방지).
FTC_OVERSAMPLE = 5
ftc_rows = [r for r in train_rows if r["provenance"] == "ftc_stage"]
train_rows = train_rows + ftc_rows * (FTC_OVERSAMPLE - 1)
random.shuffle(train_rows)
print(f"oversample 후 train: {len(train_rows)}  (ftc_stage 비중 {len(ftc_rows) * FTC_OVERSAMPLE / len(train_rows):.1%})")

train_rows[0]


## 1-1. 3차 조정 (2026-09-30) — 프롬프트 엔지니어링: 핵심주장 보존을 학습 목표에 명시

1·2차(§7, §7-1)에서 확인한 실패: 모델이 "위반을 이해해서 고치는" 게 아니라 "OOO에
도움을 줄 수 있음"이라는 고정 꼬리표를 기계적으로 붙인다 — 학습에 없던 문구("당뇨병
환자도 안심하고 드실 수 있는 치료 효과")에서 핵심 주장을 통째로 날려버렸다.

**바꾸는 것**: 정답 JSON만 바로 내놓게 하는 대신, **"핵심 주장을 먼저 밝히고 나서 그걸
지키는 JSON을 낸다"는 2단계 추론을 학습 타깃 자체에 넣는다** — 프롬프트 엔지니어링
(chain-of-thought SFT)을 재학습에 곁들이는 시도다. 데이터(821쌍)는 그대로, 학습
타깃 포맷만 바꾼다.

- `injected_golden.jsonl`(784행)에는 이미 사람이 적어 둔 `핵심주장` 필드가 있다 —
  `preprocess/inject.py`가 규칙 적용 전 원본 주장을 뽑아 둔 것. 이걸 그대로 가져다 쓴다.
- `ftc_stage` 유래 37건은 그런 필드가 없어서, 이번에 Claude가 직접 37건 전부에
  핵심주장을 새로 붙였다(아래 코드의 `FTC_CORE_CLAIMS`).
- 학습 타깃 형식: `"핵심 주장: {핵심주장}\n{RewriteSet JSON}"` — assistant가 핵심주장을
  먼저 쓰고 그다음 JSON을 내도록 강제한다. 평가(§6)에서는 마지막 `{...}` 구간만 잘라
  `RewriteSet`으로 검증하고, 핵심주장 줄은 별도로 사람이 읽어 "진짜 핵심을 짚었는지"를
  본다.


In [ ]:
GOLDEN_PATH = Path("../../data/derived/injected_golden.jsonl")  # docs/lse -> 프로젝트 루트 기준
assert GOLDEN_PATH.exists(), f"{GOLDEN_PATH} 가 없다"

core_claim_by_phrase: dict[str, str] = {}
with GOLDEN_PATH.open(encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d.get("핵심주장"):
            core_claim_by_phrase[d["문구"]] = d["핵심주장"]

# ftc_stage 행 — 원본에 핵심주장 필드가 없어 직접 붙인 값이 비공개 파일의 `core_claim` 에 있다
# (🔒 2026-10-01 — 표 자체가 광고주 문구를 인용해 노트북에서 뺐다)
for r in train_rows + held_out_rows:
    if r.get("core_claim"):
        core_claim_by_phrase[r["input"]] = r["core_claim"]

missing = 0
for r in train_rows + held_out_rows:
    cc = core_claim_by_phrase.get(r["input"])
    if cc is None:
        missing += 1
        cc = r["output"]["body"]  # 최후 대비책 — 못 찾으면 교정문 전체를 핵심주장으로
    r["core_claim"] = cc

print(f"핵심주장 매핑 못 찾은 행: {missing} / {len(train_rows) + len(held_out_rows)}")
train_rows[0]["core_claim"], held_out_rows[0]["core_claim"]


## 2. 프롬프트 구성 — 채팅 템플릿 + 정답 토큰만 loss

1차(§7-2)와 동일한 방식: system+user+assistant 채팅 템플릿을 쓰고, **assistant(정답) 토큰에만
loss를 걸고 질문 부분은 라벨을 `-100`으로 마스킹**한다.


In [ ]:
from transformers import AutoTokenizer

# 🆕 §1-1 뒤 (2026-09-30 재조정) — 1.5B 출력이 오타 많고 "OOO에 도움을 줄 수 있음"만
# 반복해 문장이 부자연스럽다는 지적 — Qwen2.5-3B-Instruct로 스케일업(기획서 §2가 이미
# 제시했던 옵션). VRAM(10GB) 여유를 위해 뒤 학습 셀에서 BATCH_SIZE=1 + gradient
# checkpointing을 쓴다.
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🆕 §1-1(3차) — 핵심주장을 먼저 밝히고, 그걸 지키면서 JSON을 내라고 명시한다.
# 1·2차는 "위반 지우고 JSON만 내라"였는데, 그게 "OOO에 도움을 줄 수 있음" 꼬리표
# 암기로 이어졌다. 이번엔 순서(핵심주장 식별 → 유지 → 위반요소만 제거)를 강제한다.
SYSTEM_PROMPT = (
    "당신은 광고 문구 준법 검수 보조 도구입니다. 위반 소지가 있는 광고 문구를 교정할 때 "
    "반드시 다음 순서를 따릅니다.\n"
    "1) 문구에서 핵심 주장(무엇에 대한 효능·성분·비교인지)을 찾는다 — 이 핵심 주장은 "
    "절대 삭제하지 않는다.\n"
    "2) 위반 요소(수치·최상급·단정 표현·부당 비교·허위 인증 등)만 정확히 찾아 제거하거나 "
    "완화한다.\n"
    "3) 핵심 주장은 그대로 남기고, 위반 요소만 뺀 자연스러운 문장으로 다시 쓴다.\n"
    "출력은 먼저 \"핵심 주장: ...\" 한 줄, 그다음 줄에 JSON만 씁니다. "
    '형식: {"body": "본문(핵심 주장 유지)", "mandatory_note": "필수 병기 문구 또는 null", '
    '"placement": "배치 지시 또는 null"}'
)


def build_user_prompt(row: dict) -> str:
    types = ", ".join(row.get("violation_types") or []) or "(미상)"
    basis = row.get("legal_basis") or "(미상)"
    return f"위반 문구: {row['input']}\n위반 유형: {types}\n근거: {basis}"


def build_target_json(row: dict) -> str:
    return json.dumps(row["output"], ensure_ascii=False)


def build_target_text(row: dict) -> str:
    """🆕 핵심주장 한 줄 + JSON — assistant가 이 순서로 생성하도록 학습시킨다."""
    return f"핵심 주장: {row['core_claim']}\n{build_target_json(row)}"


def build_messages(row: dict) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
        {"role": "assistant", "content": build_target_text(row)},
    ]


print(build_user_prompt(train_rows[0]))
print(build_target_text(train_rows[0]))


In [ ]:
MAX_LEN = 512


def tokenize_row(row: dict) -> dict:
    messages = build_messages(row)
    # 질문 부분(system+user, assistant 프롬프트 헤더까지)만 렌더 — 정답 시작 위치를 알아내기 위함
    prompt_only = tokenizer.apply_chat_template(
        messages[:2], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)

    prompt_ids = tokenizer(prompt_only, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(
        full_text, add_special_tokens=False, truncation=True, max_length=MAX_LEN
    )["input_ids"]

    labels = list(full_ids)
    n_prompt = min(len(prompt_ids), len(labels))
    for i in range(n_prompt):
        labels[i] = -100  # 질문 부분은 loss 제외 (§7-2와 동일)

    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels}


train_encoded = [tokenize_row(r) for r in train_rows]
held_out_encoded = [tokenize_row(r) for r in held_out_rows]
print("encoded:", len(train_encoded), len(held_out_encoded))
print("sample len:", len(train_encoded[0]["input_ids"]))


In [ ]:
import torch
from torch.utils.data import Dataset


class PairDataset(Dataset):
    def __init__(self, encoded_rows):
        self.rows = encoded_rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def collate(batch, pad_id):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, attn, labels = [], [], []
    for b in batch:
        pad = max_len - len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad_id] * pad)
        attn.append(b["attention_mask"] + [0] * pad)
        labels.append(b["labels"] + [-100] * pad)
    return {
        "input_ids": torch.tensor(input_ids),
        "attention_mask": torch.tensor(attn),
        "labels": torch.tensor(labels),
    }


train_ds = PairDataset(train_encoded)
held_out_ds = PairDataset(held_out_encoded)


## 3. 모델 + LoRA — `r=8, alpha=16`, `q/k/v/o_proj` (§7-2와 동일 설정)

전체 15억 파라미터 중 LoRA 어댑터 파라미터만 학습한다 — 원 가중치는 동결.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
# 🆕 3B로 올리면서 VRAM(10GB) 여유 확보 — gradient checkpointing으로 activation
# 메모리를 줄인다(속도는 조금 희생). LoRA는 입력에 requires_grad가 없어 체크포인팅과
# 충돌할 수 있어 enable_input_require_grads()를 같이 건다(peft 공식 권장 조합).
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.print_trainable_parameters()


## 4. 학습

In [ ]:
from functools import partial

from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

# 🆕 3B로 올리면서 BATCH_SIZE 2→1 (VRAM 10GB 한도, gradient checkpointing과 함께).
BATCH_SIZE = 1
# 🔴 §7(1차)에서 EPOCHS=10 이 과적합이었다 — epoch 2 avg loss가 이미 0.0111 이었고,
#    그 이상은 "위반 이해"가 아니라 "틀 암기" 구간이었다. 3으로 낮춘다.
EPOCHS = 3
LR = 2e-4

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=partial(collate, pad_id=tokenizer.pad_token_id),
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=max(1, total_steps // 10), num_training_steps=total_steps
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.train()

step = 0
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        step += 1
        if step % 100 == 0:
            print(f"step {step}  loss {loss.item():.4f}")

    print(f"epoch {epoch + 1}/{EPOCHS}  avg loss {epoch_loss / len(train_loader):.4f}")


## 5. 어댑터 저장

`copylane_sllm_lora_adapter/`(이 노트북과 같은 폴더)에 저장한다. 학습이 끝나면 이 폴더를
프로젝트 `models/`로 옮긴다 — `models/`는 gitignore 대상이라(`app/encoder.py`가 로딩하는
KC-BERT와 동일한 방식) 커밋되지 않는다.


In [ ]:
ADAPTER_DIR = Path("copylane_sllm_lora_adapter")
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"saved to {ADAPTER_DIR.resolve()}")
print("다음: 이 폴더를 프로젝트 models/copylane_sllm_lora_adapter/ 로 이동")


## 6. held-out 평가 — `RewriteSet` 스키마 검증

§7-4가 명시한 대로 "증명하지 않는 것"은 품질·일반화다. 이 평가는 **형식 정확도**
(생성물이 `RewriteSet`으로 파싱되는지)만 본다 — 내용 품질 평가는 범위 밖.


In [ ]:
import sys

# app.contracts.RewriteSet 재사용 — 프로젝트 루트를 sys.path에 추가
PROJECT_ROOT = Path.cwd().resolve().parents[1]  # docs/lse -> 프로젝트 루트
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.contracts import RewriteSet  # noqa: E402

model.eval()


def generate(row: dict) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device if hasattr(model, "device") else device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220, do_sample=False)
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


def split_reasoning_and_json(raw: str) -> tuple[str, str]:
    """🆕 "핵심 주장: ...\\n{json}" 형식에서 핵심주장 줄과 JSON을 분리한다."""
    idx = raw.find("{")
    if idx == -1:
        return raw.strip(), ""
    return raw[:idx].strip(), raw[idx:].strip()


n_valid = 0
samples = []
for row in held_out_rows:
    raw = generate(row)
    reasoning, json_part = split_reasoning_and_json(raw)
    try:
        parsed = RewriteSet.model_validate_json(json_part)
        ok = True
        n_valid += 1
    except Exception as e:  # noqa: BLE001 — 형식 검증 실패 사례를 그대로 보여주기 위함
        parsed = None
        ok = False
    samples.append({"input": row["input"], "expected_body": row["output"]["body"],
                     "expected_core": row["core_claim"], "reasoning": reasoning,
                     "raw": raw, "schema_valid": ok, "provenance": row["provenance"]})

print(f"schema-valid 전체: {n_valid}/{len(held_out_rows)} ({n_valid / len(held_out_rows):.1%})")
print()
print("=== injected_golden held_out (처음 5개) ===")
for s in [s for s in samples if s["provenance"] != "ftc_stage"][:5]:
    print(s)
print()
print("=== ftc_stage 실사례 held_out (전부) — 핵심주장 보존이 이번엔 되는지 직접 확인 ===")
for s in [s for s in samples if s["provenance"] == "ftc_stage"]:
    print(f"입력: {s['input']}")
    print(f"기대 핵심주장: {s['expected_core']}  |  모델이 밝힌 핵심주장: {s['reasoning']}")
    print(f"기대(직접 작성): {s['expected_body']}")
    print(f"모델 생성: {s['raw']}  [{'VALID' if s['schema_valid'] else 'INVALID'}]")
    print()


## 7. 결과 요약 (2026-09-22 로컬 실행 완료)

| | 1차 Colab (35쌍, 직접작성) | 2차 로컬 (806쌍, injected_golden+ftc_stage) |
|---|--:|--:|
| 학습 loss | 1.44 → 0.87 | **0.1275 → 0.0003** (epoch 1→10 평균) |
| held-out 스키마 통과율 | 5/5 | **120/120 (100%)** |
| held-out 중 실사례(`ftc_stage`, 1건) | — | 스키마는 통과, **내용은 아래 참고** |
| 데이터 출처 | 직접 작성(합성) | 규칙기반 주입(틀 다양성 낮음) + 실제 의결서(수량 적음, 교정문 미검토) |

### 🔴 증명하지 않는 것 — 이번엔 특히 중요

**스키마 통과율 100%가 품질을 뜻하지 않는다.** loss가 epoch 3만에 0.005 밑으로 떨어진 건
정상적인 수렴이 아니라 **과적합 신호였다** — §1에서 미리 경고한 "틀 다양성 낮음"이 그대로
현실화됐다. adapter로 직접 테스트해보면:

```
[학습에 없던 실사례] "〔공정위 인용 문구 A — 업력·업계 순위 주장 · 비공개〕"
  기대(내가 작성): "〔A 교정문 — 순위를 빼고 판매 사실만 · 비공개〕"
  모델 생성:        "〔A에서 '체지방'이 빠지고 꼬리표만 붙음〕"   ← "체지방"이 빠지고 문법이 깨짐

[완전히 새로운 문구] "당뇨병 환자도 안심하고 드실 수 있는 치료 효과"
  모델 생성: "치료 효과에 도움을 줄 수 있음"   ← 핵심 주장("당뇨병 환자도 안심")을 통째로 버림
```

모델이 위반을 이해하고 고치는 게 아니라, `injected_golden`의 T1 규칙이 압도적으로 많이
준 **"OOO에 도움을 줄 수 있음"이라는 고정 꼬리표를 거의 모든 입력에 기계적으로 붙이는 법**을
배웠다 — 원문의 핵심 내용을 그대로 보존하는 능력이 약하다. `preprocess/inject.py` 원작성자가
경고한 "위반이 아니라 틀을 배운다"가 정확히 이 실행에서 재현됐다.

**증명하는 것**: KC-BERT 위반 후보 추출 → sLLM 재작성 → `RewriteSet` 스키마 검증까지 **파이프라인
배선 자체는 끝까지 돈다** — MCP 시연에서 "형식이 갖춰진 JSON이 나온다"는 보여줄 수 있다.
**증명하지 않는 것**: 재작성 내용의 품질·원문 보존 — 지금 이 어댑터를 실제 판정 결과에
그대로 노출하면 안 된다. 다음 라운드는 데이터 틀 다양성을 늘리는 것이 학습 설정보다 먼저다.
